In [1]:

%pip install -q \
    "ragas==0.3.9" \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-chroma==0.2.6" \
    "langchain-text-splitters==0.3.9" \
    "transformers==4.56.2" \
    "sentence-transformers" \
    "accelerate" \
    "datasets" \
    "pandas" \
    "scikit-learn"


In [2]:

import torch
import pandas as pd
import numpy as np
import ragas
import langchain

from datasets import Dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Ragas:", ragas.__version__)
print("LangChain:", langchain.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Imports successful!")


Ragas: 0.3.9
LangChain: 0.3.27
PyTorch: 2.11.0+cpu
GPU available: False
Imports successful!


In [16]:

knowledge_docs = [
    "Machine learning enables computers to learn patterns from data and make predictions or decisions.",
    "Supervised learning trains a model using labeled examples containing inputs and known outputs.",
    "Unsupervised learning discovers patterns or groups in data without labeled target outputs.",
    "A Retrieval-Augmented Generation (RAG) system retrieves relevant information and supplies it to a language model to generate a grounded answer.",
    "A vector database stores and searches vector embeddings to retrieve semantically similar information.",
    "Embeddings are numerical vector representations of data that capture semantic or contextual information.",
    "Chunking divides documents into smaller sections to improve retrieval and fit information into a model's context.",
    "Precision measures the proportion of predicted positive examples that are actually positive.",
    "Recall measures the proportion of actual positive examples that are correctly identified.",
    "A train-test split separates data into training and testing subsets to assess model generalization.",
    "Faithfulness measures whether an answer is supported by the retrieved context.",
    "Context precision measures whether relevant retrieved information is ranked above irrelevant information.",
    "Context recall measures whether the retrieved context contains the information needed to answer the question."
]

documents = [
    Document(page_content=text, metadata={"source": f"doc_{i+1}"})
    for i, text in enumerate(knowledge_docs)
]

print("Knowledge documents:", len(documents))
print("Sample document:", documents[0].page_content)


Knowledge documents: 13
Sample document: Machine learning enables computers to learn patterns from data and make predictions or decisions.


In [17]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)


Original documents: 13
Total chunks: 13

First chunk:
Machine learning enables computers to learn patterns from data and make predictions or decisions.


In [18]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={"normalize_embeddings": True}
)

test_embedding = embedding_model.embed_query(
    "What is machine learning?"
)

print("Embedding model loaded successfully.")
print("Embedding dimensions:", len(test_embedding))
print("GPU available:", torch.cuda.is_available())


Embedding model loaded successfully.
Embedding dimensions: 384
GPU available: False


In [19]:

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="w11d4_rag_eval_fresh",
    persist_directory="./chroma_w11d4_fresh"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

test_question = "What is machine learning?"
retrieved_docs = retriever.invoke(test_question)

print("Vector database created.")
print("Retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nChunk {i}:")
    print(doc.page_content)


Vector database created.
Retrieved chunks: 3

Chunk 1:
Machine learning enables computers to learn patterns from data and make predictions or decisions.

Chunk 2:
Machine learning enables computers to learn patterns from data and make predictions or decisions.

Chunk 3:
Machine learning enables computers to learn patterns from data and make predictions or decisions.


In [20]:

%pip install -q --force-reinstall --no-deps "transformers==4.56.2"


In [21]:

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

local_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

local_model.to("cpu")
local_model.eval()

print("Local instruction model loaded successfully.")
print("Model:", MODEL_NAME)
print("Device: CPU")


Local instruction model loaded successfully.
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: CPU


In [22]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

local_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

local_model.to("cpu")
local_model.eval()

print("Local instruction model loaded successfully.")
print("Model:", MODEL_NAME)
print("Device: CPU")


Local instruction model loaded successfully.
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: CPU


In [23]:

def generate_local_answer(prompt, max_new_tokens=160):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer accurately using only the supplied context. "
                "Be concise. Do not invent facts."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():
        outputs = local_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


print(generate_local_answer(
    "Reply with exactly: Local model test successful."
))



The local model has passed its test successfully.


In [24]:

def rag_answer(question, current_retriever):
    retrieved_docs = current_retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
Use the context below to answer the question.
If the context does not contain the answer, say you do not know.
Give a short, direct answer.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_local_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "contexts": [
            doc.page_content for doc in retrieved_docs
        ]
    }


print("RAG answer function defined.")


RAG answer function defined.


In [25]:

qa_data = [
    {
        "question": "What is machine learning?",
        "ground_truth": "Machine learning enables computers to learn patterns from data and make predictions or decisions."
    },
    {
        "question": "What is supervised learning?",
        "ground_truth": "Supervised learning trains a model using labeled examples containing inputs and known outputs."
    },
    {
        "question": "What is unsupervised learning?",
        "ground_truth": "Unsupervised learning discovers patterns or groups in data without labeled target outputs."
    },
    {
        "question": "What is a RAG system?",
        "ground_truth": "RAG retrieves relevant information and supplies it to a language model to generate a grounded answer."
    },
    {
        "question": "What is a vector database?",
        "ground_truth": "A vector database stores and searches vector embeddings to retrieve semantically similar information."
    },
    {
        "question": "What are embeddings?",
        "ground_truth": "Embeddings are numerical vector representations of data that capture semantic or contextual information."
    },
    {
        "question": "Why is chunking used in RAG?",
        "ground_truth": "Chunking divides documents into smaller sections to improve retrieval and fit information into a model's context."
    },
    {
        "question": "What does precision measure?",
        "ground_truth": "Precision measures the proportion of predicted positive examples that are actually positive."
    },
    {
        "question": "What does recall measure?",
        "ground_truth": "Recall measures the proportion of actual positive examples that are correctly identified."
    },
    {
        "question": "What is a train-test split?",
        "ground_truth": "A train-test split separates data into training and testing subsets to assess model generalization."
    }
]

qa_df = pd.DataFrame(qa_data)

print("Evaluation questions:", len(qa_df))
display(qa_df)


Evaluation questions: 10


,question,ground_truth
0,What is machine learning?,Machine learning enables computers to learn pa...
1,What is supervised learning?,Supervised learning trains a model using label...
2,What is unsupervised learning?,Unsupervised learning discovers patterns or gr...
3,What is a RAG system?,RAG retrieves relevant information and supplie...
4,What is a vector database?,A vector database stores and searches vector e...
5,What are embeddings?,Embeddings are numerical vector representation...
6,Why is chunking used in RAG?,Chunking divides documents into smaller sectio...
7,What does precision measure?,Precision measures the proportion of predicted...
8,What does recall measure?,Recall measures the proportion of actual posit...
9,What is a train-test split?,A train-test split separates data into trainin...


In [26]:

results = []

for i, item in enumerate(qa_data, start=1):
    print(f"Processing question {i}/{len(qa_data)}...")

    result = rag_answer(item["question"], retriever)

    result["ground_truth"] = item["ground_truth"]
    results.append(result)

evaluation_df = pd.DataFrame(results)

evaluation_df.to_csv(
    "w11d4_rag_answers_initial.csv",
    index=False
)

print("\nInitial RAG answers generated.")
print("Rows:", len(evaluation_df))
print("CSV saved: w11d4_rag_answers_initial.csv")

display(evaluation_df[["question", "answer", "ground_truth"]])


Processing question 1/10...
Processing question 2/10...
Processing question 3/10...
Processing question 4/10...
Processing question 5/10...
Processing question 6/10...
Processing question 7/10...
Processing question 8/10...
Processing question 9/10...
Processing question 10/10...

Initial RAG answers generated.
Rows: 10
CSV saved: w11d4_rag_answers_initial.csv


,question,answer,ground_truth
0,What is machine learning?,Machine learning enables computers to learn pa...,Machine learning enables computers to learn pa...
1,What is supervised learning?,Supervised learning trains a model using label...,Supervised learning trains a model using label...
2,What is unsupervised learning?,Unsupervised learning discovers patterns or gr...,Unsupervised learning discovers patterns or gr...
3,What is a RAG system?,A Retrieval-Augmented Generation (RAG) system ...,RAG retrieves relevant information and supplie...
4,What is a vector database?,A vector database is a database that stores an...,A vector database stores and searches vector e...
5,What are embeddings?,Embeddings are numerical vector representation...,Embeddings are numerical vector representation...
6,Why is chunking used in RAG?,To improve retrieval and fit information into ...,Chunking divides documents into smaller sectio...
7,What does precision measure?,Precision measures the proportion of predicted...,Precision measures the proportion of predicted...
8,What does recall measure?,The recall measure measures the proportion of ...,Recall measures the proportion of actual posit...
9,What is a train-test split?,A train-test split separates data into trainin...,A train-test split separates data into trainin...


In [27]:
from langchain_core.language_models.llms import LLM
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

class LocalQwenLLM(LLM):
    @property
    def _llm_type(self):
        return "local-qwen"

    @property
    def _identifying_params(self):
        return {"model": MODEL_NAME}

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        answer = generate_local_answer(prompt, max_new_tokens=256)

        if stop:
            for stop_text in stop:
                if stop_text in answer:
                    answer = answer.split(stop_text)[0]

        return answer.strip()

local_llm = LocalQwenLLM()
evaluator_llm = LangchainLLMWrapper(local_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

print("Local Qwen evaluator created.")

Local Qwen evaluator created.


/tmp/ipykernel_47231/520978288.py:25: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(local_llm)
/tmp/ipykernel_47231/520978288.py:26: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)


In [35]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = evaluator_llm

    if hasattr(metric, "embeddings"):
        metric.embeddings = evaluator_embeddings

print("Metric configuration:")

for metric in metrics:
    llm_ready = getattr(metric, "llm", None) is not None
    embeddings_ready = (
        getattr(metric, "embeddings", None) is not None
    )

    print(
        f"{metric.name} | "
        f"LLM configured: {llm_ready} | "
        f"Embeddings configured: {embeddings_ready}"
    )

print("Configuration cell completed.")

Metric configuration:
faithfulness | LLM configured: True | Embeddings configured: False
answer_relevancy | LLM configured: True | Embeddings configured: True
context_precision | LLM configured: True | Embeddings configured: False
context_recall | LLM configured: True | Embeddings configured: False
Configuration cell completed.


In [36]:
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig

ragas_dataset = Dataset.from_dict({
    "question": evaluation_df["question"].tolist(),
    "answer": evaluation_df["answer"].tolist(),
    "contexts": evaluation_df["contexts"].tolist(),
    "ground_truth": evaluation_df["ground_truth"].tolist()
})

test_dataset = ragas_dataset.select(range(2))

print("Full dataset rows:", len(ragas_dataset))
print("Test dataset rows:", len(test_dataset))

test_result = evaluate(
    dataset=test_dataset,
    metrics=[faithfulness],
    run_config=RunConfig(
        timeout=300,
        max_retries=1,
        max_workers=1
    ),
    raise_exceptions=True
)

display(test_result.to_pandas())

Full dataset rows: 10
Test dataset rows: 2


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness
0,What is machine learning?,[Machine learning enables computers to learn p...,Machine learning enables computers to learn pa...,Machine learning enables computers to learn pa...,1.0
1,What is supervised learning?,[Supervised learning trains a model using labe...,Supervised learning trains a model using label...,Supervised learning trains a model using label...,1.0


In [37]:
initial_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    run_config=RunConfig(
        timeout=300,
        max_retries=1,
        max_workers=1
    ),
    raise_exceptions=True
)

initial_scores_df = initial_result.to_pandas()

display(
    initial_scores_df[
        ["faithfulness", "answer_relevancy",
         "context_precision", "context_recall"]
    ].describe()
)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

,faithfulness,answer_relevancy,context_precision,context_recall
count,10.000000,10.000000,10.000000,10.0
mean,0.900000,0.833589,0.900000,1.0
std,0.210819,0.274849,0.316228,0.0
min,0.500000,0.127745,0.000000,1.0
25%,1.000000,0.833881,1.000000,1.0
50%,1.000000,0.950399,1.000000,1.0
75%,1.000000,0.993444,1.000000,1.0
max,1.000000,1.000000,1.000000,1.0


In [38]:
initial_scores_df.to_csv(
    "w11d4_ragas_initial_scores.csv",
    index=False
)

print("Initial Ragas scores saved.")
print("File: w11d4_ragas_initial_scores.csv")
print("Rows:", len(initial_scores_df))

display(initial_scores_df.head())

Initial Ragas scores saved.
File: w11d4_ragas_initial_scores.csv
Rows: 10


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is machine learning?,[Machine learning enables computers to learn p...,Machine learning enables computers to learn pa...,Machine learning enables computers to learn pa...,1.0,0.810880,1.0,1.0
1,What is supervised learning?,[Supervised learning trains a model using labe...,Supervised learning trains a model using label...,Supervised learning trains a model using label...,1.0,0.932798,1.0,1.0
2,What is unsupervised learning?,[Unsupervised learning discovers patterns or g...,Unsupervised learning discovers patterns or gr...,Unsupervised learning discovers patterns or gr...,1.0,0.902885,1.0,1.0
3,What is a RAG system?,[A Retrieval-Augmented Generation (RAG) system...,A Retrieval-Augmented Generation (RAG) system ...,RAG retrieves relevant information and supplie...,0.5,0.619802,1.0,1.0
4,What is a vector database?,[A vector database stores and searches vector ...,A vector database is a database that stores an...,A vector database stores and searches vector e...,1.0,1.000000,1.0,1.0


In [39]:
metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

initial_summary = initial_scores_df[metric_names].mean().sort_values()

display(initial_summary.to_frame("Initial Mean Score"))

weakest_metric = initial_summary.index[0]
print("Lowest-scoring metric:", weakest_metric)

,Initial Mean Score
answer_relevancy,0.833589
context_precision,0.900000
faithfulness,0.900000
context_recall,1.000000


Lowest-scoring metric: answer_relevancy


In [40]:
optimized_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Original retrieval k:", 3)
print("Optimized retrieval k:", 5)

Original retrieval k: 3
Optimized retrieval k: 5


In [41]:
optimized_results = []

for i, item in enumerate(qa_data, start=1):
    print(f"Optimizing question {i}/{len(qa_data)}...")

    result = rag_answer(
        item["question"],
        optimized_retriever
    )

    result["ground_truth"] = item["ground_truth"]
    optimized_results.append(result)

optimized_df = pd.DataFrame(optimized_results)

optimized_df.to_csv(
    "w11d4_rag_answers_optimized.csv",
    index=False
)

print("Optimized answers generated:", len(optimized_df))
display(optimized_df[["question", "answer"]].head())

Optimizing question 1/10...
Optimizing question 2/10...
Optimizing question 3/10...
Optimizing question 4/10...
Optimizing question 5/10...
Optimizing question 6/10...
Optimizing question 7/10...
Optimizing question 8/10...
Optimizing question 9/10...
Optimizing question 10/10...
Optimized answers generated: 10


,question,answer
0,What is machine learning?,Machine learning is the enabling technology th...
1,What is supervised learning?,Supervised learning trains a model using label...
2,What is unsupervised learning?,Unsupervised learning discovers patterns or gr...
3,What is a RAG system?,A Retrieval-Augmented Generation (RAG) system ...
4,What is a vector database?,A vector database is a database that stores an...


In [42]:
optimized_dataset = Dataset.from_dict({
    "question": optimized_df["question"].tolist(),
    "answer": optimized_df["answer"].tolist(),
    "contexts": optimized_df["contexts"].tolist(),
    "ground_truth": optimized_df["ground_truth"].tolist()
})

print("Optimized dataset rows:", len(optimized_dataset))

Optimized dataset rows: 10


In [43]:
optimized_result = evaluate(
    dataset=optimized_dataset,
    metrics=metrics,
    run_config=RunConfig(
        timeout=300,
        max_retries=1,
        max_workers=1
    ),
    raise_exceptions=True
)

optimized_scores_df = optimized_result.to_pandas()

optimized_scores_df.to_csv(
    "w11d4_ragas_optimized_scores.csv",
    index=False
)

print("Optimized Ragas evaluation completed.")
display(optimized_scores_df[metric_names].mean().to_frame("Optimized Mean Score"))

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Optimized Ragas evaluation completed.


,Optimized Mean Score
faithfulness,1.000000
answer_relevancy,0.791472
context_precision,0.932500
context_recall,1.000000


In [44]:
metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

comparison_df = pd.DataFrame({
    "Initial Mean": initial_scores_df[metric_names].mean(),
    "Optimized Mean": optimized_scores_df[metric_names].mean()
})

comparison_df["Improvement"] = (
    comparison_df["Optimized Mean"] - comparison_df["Initial Mean"]
)

display(comparison_df.round(4))

,Initial Mean,Optimized Mean,Improvement
faithfulness,0.9000,1.0000,0.1000
answer_relevancy,0.8336,0.7915,-0.0421
context_precision,0.9000,0.9325,0.0325
context_recall,1.0000,1.0000,0.0000


In [45]:
comparison_df.to_csv("w11d4_ragas_comparison.csv")

print("Comparison report saved.")
display(comparison_df.round(4))

Comparison report saved.


,Initial Mean,Optimized Mean,Improvement
faithfulness,0.9000,1.0000,0.1000
answer_relevancy,0.8336,0.7915,-0.0421
context_precision,0.9000,0.9325,0.0325
context_recall,1.0000,1.0000,0.0000


In [46]:
import os

output_files = [
    "w11d4_rag_answers_initial.csv",
    "w11d4_ragas_initial_scores.csv",
    "w11d4_rag_answers_optimized.csv",
    "w11d4_ragas_optimized_scores.csv",
    "w11d4_ragas_comparison.csv"
]

for filename in output_files:
    print(f"{filename}: {'READY' if os.path.exists(filename) else 'MISSING'}")

w11d4_rag_answers_initial.csv: READY
w11d4_ragas_initial_scores.csv: READY
w11d4_rag_answers_optimized.csv: READY
w11d4_ragas_optimized_scores.csv: READY
w11d4_ragas_comparison.csv: READY


In [47]:
from google.colab import files

for filename in output_files:
    if os.path.exists(filename):
        files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>